In [1]:
import arxiv
import os
import json
from typing import List
from dotenv import load_dotenv
import anthropic

In [2]:
_ = load_dotenv()

In [4]:
PAPER_DIR = 'papers'

In [5]:
def search_papers(topic, max_results = 5):
    client = arxiv.Client()
    search = arxiv.Search(
        query = topic,
        max_results = max_results,
        sort_by = arxiv.SortCriterion.Relevance
    )
    papers = client.results(search)

    path = os.path.join(PAPER_DIR, topic.lower().replace(' ', '_'))
    os.makedirs(path,exist_ok=True)

    file_path = os.path.join(path,'papers_info.json')

    try:
        with open(file_path,'r') as json_file:
            papers_info = json.load(json_file)
    except (FileNotFoundError,json.JSONDecodeError):
        papers_info = {}
    
    paper_ids = []
    for paper in papers:
        paper_ids.append(paper.get_short_id())
        paper_info = {
            'title' : paper.title,
            'authors' : [author.name for author in paper.authors],
            'summary' : paper.summary,
            'pdf_url' : paper.pdf_url,
            'published' : str(paper.published.date())
        }
        papers_info[paper.get_short_id()] = paper_info

    with open(file_path,'w') as json_file:
        json.dump(papers_info, json_file, indent = 2)
    
    print(f'results saved at {file_path}')

    return paper_ids

In [6]:
search_papers('transformers')

results saved at papers/transformers/papers_info.json


['2512.22190v1',
 '2512.22189v1',
 '2209.07474v3',
 '2506.22084v1',
 '2308.07110v1']

In [7]:
def extract_info(paper_id):
    for item in os.listdir(PAPER_DIR):
        item_path = os.path.join(PAPER_DIR,item)
        if os.path.isdir(item_path):
            file_path = os.path.join(item_path,'papers_info.json')
            if os.path.isfile(file_path):
                try:
                    with open(file_path,'r') as json_file:
                        papers_info = json.load(json_file)
                        if paper_id in papers_info:
                            return json.dumps(papers_info[paper_id],indent = 2)
                except (FileNotFoundError,json.JSONDecodeError) as e:
                    print(f'Error loading {file_path} : {str(e)}')
                    continue
    return f'Paper {paper_id} does not exist'

In [8]:
extract_info('2512.22190v1')

'{\n  "title": "Physics-Informed Machine Learning for Transformer Condition Monitoring -- Part I: Basic Concepts, Neural Networks, and Variants",\n  "authors": [\n    "Jose I. Aizpurua"\n  ],\n  "summary": "Power transformers are critical assets in power networks, whose reliability directly impacts grid resilience and stability. Traditional condition monitoring approaches, often rule-based or purely physics-based, struggle with uncertainty, limited data availability, and the complexity of modern operating conditions. Recent advances in machine learning (ML) provide powerful tools to complement and extend these methods, enabling more accurate diagnostics, prognostics, and control. In this two-part series, we examine the role of Neural Networks (NNs) and their extensions in transformer condition monitoring and health management tasks. This first paper introduces the basic concepts of NNs, explores Convolutional Neural Networks (CNNs) for condition monitoring using diverse data modalities